# 24 · Silver: match ICA events to disaster events
**Owner: M** · Layer: silver · Runs after: 21, 22

**What:** link each ICA event to at most one disaster event, so gold can borrow deaths, injuries, evacuations and homes destroyed.

**Rule:** same state (any state in the disaster event's list), same hazard group, start dates within **7 days**. Best candidate = smallest day gap, then most similar name. Matching is **one-to-one**: a disaster event can't be claimed by two ICA events.

**Why it is checked against insured cost:** profiling showed the disaster-events insured cost is usually ICA's own figure. Where both exist, agreement is independent evidence that a match is right; a large disagreement flags a likely wrong match.

In [ ]:
%run ./00_common

In [ ]:
MAX_DAY_GAP = 7
ica = read("silver", "ica_events").where("state_code IS NOT NULL AND hazard_group IS NOT NULL")
de = read("silver", "disaster_events").where("start_date IS NOT NULL AND natural_hazard")
de_states = read("silver", "disaster_event_states").where("state_code IS NOT NULL")
de_x = de.join(de_states, "de_id").select("de_id", "state_code", "hazard_group", F.col("start_date").alias("de_start"),
                                          "title", "title_place", "insured_cost_aud")

cand = (ica.select("event_id", "event_name", "state_code", "hazard_group", "start_date", "original_loss_aud")
        .join(de_x, ["state_code", "hazard_group"])
        .withColumn("day_gap", F.abs(F.datediff("de_start", "start_date")))
        .where(F.col("day_gap") <= MAX_DAY_GAP)
        .withColumn("name_distance", F.levenshtein(F.lower("event_name"), F.lower("title_place")))
        .dropDuplicates(["event_id", "de_id"]))

# best disaster event for each ICA event, then best ICA event for each disaster event (one-to-one)
w_ica = Window.partitionBy("event_id").orderBy("day_gap", "name_distance", "de_id")
w_de = Window.partitionBy("de_id").orderBy("day_gap", "name_distance", "event_id")
best = cand.withColumn("r1", F.row_number().over(w_ica)).where("r1 = 1").withColumn("r2", F.row_number().over(w_de))
dropped_many_to_one = best.where("r2 > 1").count()
match = (best.where("r2 = 1")
         .withColumn("cost_ratio", F.when(F.col("original_loss_aud") > 0, F.round(F.col("insured_cost_aud") / F.col("original_loss_aud"), 2)))
         .withColumn("cost_check", F.when(F.col("cost_ratio").isNull(), "no figure to compare")
                     .when(F.col("cost_ratio").between(0.95, 1.05), "agrees (±5%)")
                     .when(F.col("cost_ratio").between(0.5, 2.0), "differs moderately")
                     .otherwise("differs widely: check match"))
         .withColumn("match_confidence", F.when(F.col("cost_check") == "differs widely: check match", "low")
                     .when((F.col("day_gap") <= 1) | (F.col("cost_check") == "agrees (±5%)"), "high").otherwise("medium"))
         .select("event_id", "de_id", "event_name", "title", "start_date", "de_start", "state_code", "hazard_group",
                 "day_gap", "name_distance", "original_loss_aud", "insured_cost_aud", "cost_ratio", "cost_check", "match_confidence"))
n = write(match, "silver", "ica_de_match")
log_step("silver", "match_events", "silver.ica_de_match", cand.select("event_id").distinct().count(), n, dropped_many_to_one,
         "rows_in = ICA events with at least one candidate; rejected = lost the one-to-one tie-break")

In [ ]:
m = read("silver", "ica_de_match")
display(m.groupBy("cost_check").count())
display(m.groupBy("match_confidence").count())
display(m.join(read("silver", "ica_events").select("event_id", "in_scope"), "event_id").groupBy("in_scope").count())
display(m.where("match_confidence = 'low'").select("event_name", "title", "start_date", "de_start", "cost_ratio"))